In [ ]:
import sys, os, pickle, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, mannwhitneyu, kruskal, fisher_exact
from statsmodels.stats.multitest import multipletests

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
from spk_lfp_cluster_comp_analysis import *
from spk_feat_cluster_comp_analysis import compile_experiment_results
from config import SPE1_PICKLE_ROOT, PRIORITY_CELLS, CELL_IDS, DICT_CELL_TYPE, DICT_PATCH_TYPE, DICT_CORT_DEPTH, DICT_DARK_NEURONS, DICT_EAP_WAV

warnings.filterwarnings("ignore")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
SLIDING_DIR  = os.path.join(SPE1_PICKLE_ROOT, "lfp_spk_group_pickles")
PREPOST_DIR  = os.path.join(SPE1_PICKLE_ROOT, "prepost_specparam_pickles")
CLUSTER_DIR  = os.path.join(SPE1_PICKLE_ROOT, "cluster_pickles")

# ── Cell subsets ───────────────────────────────────────────────────────────────
subsets = get_cell_subsets(PRIORITY_CELLS, CELL_IDS)
SUBSET_LABELS = [
    ("Priority cells",      subsets["priority"]),
    ("Non-priority cells",  subsets["np"]),
    ("All cells combined",  subsets["all"]),
]
print(f"Priority: {len(subsets['priority'])} | NP: {len(subsets['np'])} | All: {len(subsets['all'])}")

# ── Feature type definitions ───────────────────────────────────────────────────
# Numerical LFP features (effect sizes, continuous)
LFP_NUMERICAL = ["exponent", "r_squared", "band_aucs.theta",
                  "band_aucs.gamma", "lfp_mean"]
# Dropped as redundant: offset (collinear w/ exponent), lfp_std, lfp_exponent

# Spike feature colours
feature_shades = {
    'peak_amp_cluster':         '#8c564b',
    'peak_sharpness_cluster':   '#a06d62',
    'peak_width_cluster':       '#b38479',
    'exp_lambda_cluster':       '#c561a8',
    'inflection_time_cluster':  '#9b59b6',
    'exp_const_cluster':        '#d7aee0',
    'log_isi_cluster':          '#7f7f7f',
    'spk_times_ms_cluster':     '#b0b0b0',
}

# Module-level constants (not exported by import *)
_CB_PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7',
               '#56B4E9', '#E69F00', '#F0E442', '#000000']
_SIG_COL    = '#D55E00'
_INSIG_COL  = '#56B4E9'
_FS_SM, _FS_AX, _FS_SUB, _FS_TTL = 11, 13, 14, 16


## Load data

Three data sources get merged here:
1. **Spike cluster master table** — nRMSE, cos_sim, num_clusters, temporal drift per (cell, spike_feature)
2. **LFP sliding window stats** — Cohen's d, % significant windows per (cell, spike_feature, lfp_feature)
3. **LFP pre/post stats** — Hedges' g, d_z per (cell, spike_feature, lfp_feature, window)

Cell metadata (cell type, patch type, depth, etc.) is added from config.

In [ ]:
df_pop_stats, pop_traces = compile_lfp_stats(SLIDING_DIR)
df_prepost               = compile_prepost_stats(PREPOST_DIR)
df_master                = compile_experiment_results(CLUSTER_DIR)

# Summarise sliding stats to one row per (cell, spike_feature, lfp_feature)
df_lfp_summary = summarise_cell_lfp_effects(
    df_pop_stats[df_pop_stats["lfp_feature"].isin(LFP_NUMERICAL)])

# Add cell metadata
def add_metadata(df):
    df = df.copy()
    df["cell_num"]    = df["cell_id"].str.lstrip("c").astype(int)
    df["cell_type"]   = df["cell_num"].map(DICT_CELL_TYPE)    # PC / IN — categorical
    df["patch_type"]  = df["cell_num"].map(DICT_PATCH_TYPE)   # Juxta/WC, IC/VC — categorical
    df["cort_depth"]  = df["cell_num"].map(DICT_CORT_DEPTH)   # µm — numerical
    df["dark_neuron"] = df["cell_num"].map(DICT_DARK_NEURONS)  # bool — categorical
    df["eap_visible"] = df["cell_num"].map(DICT_EAP_WAV)      # bool — categorical
    df["clamp_mode"]  = df["patch_type"].str.split(",").str[-1].str.strip()  # IC / VC
    df["recording_type"] = df["patch_type"].str.split(",").str[0].str.strip()  # Juxta / WC
    return df

df_lfp_summary = add_metadata(df_lfp_summary)
df_prepost_meta = add_metadata(
    df_prepost[df_prepost["lfp_feature"].isin(LFP_NUMERICAL)].copy())
df_master_meta  = add_metadata(df_master.copy())

# Spike feature name normalisation for merging
df_lfp_summary["spike_feature_base"] = df_lfp_summary["spike_feature"].str.replace("_cluster","",regex=False)
print(f"LFP summary: {df_lfp_summary['cell_id'].nunique()} cells")
print(f"Pre/post:    {df_prepost_meta['cell_id'].nunique()} cells")
print(f"Master:      {df_master_meta['cell_id'].nunique()} cells")

## Does metadata predict spike clustering?

**Categorical metadata × categorical spike outcome** → Fisher's exact / Chi-square.
**Numerical metadata (depth) × numerical spike outcome (nRMSE, cos_sim)** → Spearman ρ + BH-FDR.

We ask: do PCs vs INs, Juxta vs WC, IC vs VC, or cortical depth predict whether a spike feature clusters and how different the waveforms are?

In [ ]:
from scipy.stats import chi2_contingency

cat_meta = ["cell_type", "recording_type", "clamp_mode", "dark_neuron", "eap_visible"]
num_meta = ["cort_depth"]
spk_num  = ["nRMSE", "cos_sim", "temporal_rho"]

# Per cell: one row with has_clusters, mean nRMSE, mean cos_sim
df_cell = (df_master_meta
    .groupby(["cell_id", "cell_type", "recording_type", "clamp_mode",
              "dark_neuron", "eap_visible", "cort_depth"])
    .agg(nRMSE=("nRMSE","mean"), cos_sim=("cos_sim","mean"),
         temporal_rho=("temporal_rho","mean"),
         num_features=("spike_feature","nunique"))
    .reset_index())

print("--- Categorical metadata vs spike outcome ---")
rows = []
for meta in cat_meta:
    for spk in spk_num:
        d = df_cell.dropna(subset=[meta, spk])
        groups = d.groupby(meta)[spk].apply(list)
        if len(groups) >= 2:
            stat, p = kruskal(*groups.values)
            rows.append({"meta": meta, "spike_outcome": spk,
                         "test": "Kruskal-Wallis", "stat": stat, "p": p})
for meta in num_meta:
    for spk in spk_num:
        d = df_cell.dropna(subset=[meta, spk])
        r, p = spearmanr(d[meta], d[spk])
        rows.append({"meta": meta, "spike_outcome": spk,
                     "test": "Spearman", "stat": r, "p": p})

df_meta_spk = pd.DataFrame(rows)
_, df_meta_spk["p_fdr"], _, _ = multipletests(
    df_meta_spk["p"].fillna(1), method="fdr_bh")
df_meta_spk["sig"] = df_meta_spk["p_fdr"].apply(
    lambda p: "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns")
display(df_meta_spk.sort_values("p_fdr"))

## Does metadata predict LFP effect strength?

**Categorical metadata × numerical LFP outcome (max |Cohen's d|)** → Mann-Whitney / Kruskal-Wallis + bootstrap CI + BH-FDR.
**Numerical metadata (depth) × numerical LFP outcome** → Spearman ρ + BH-FDR.

In [ ]:
rows = []
for label, cell_ids in SUBSET_LABELS:
    df_sub = df_lfp_summary[df_lfp_summary["cell_id"].isin(cell_ids)]

    for meta in cat_meta:
        for lfp in LFP_NUMERICAL:
            d = df_sub[df_sub["lfp_feature"] == lfp].dropna(subset=[meta, "max_abs_cohens_d"])
            groups = d.groupby(meta)["max_abs_cohens_d"].apply(list)
            if len(groups) >= 2 and all(len(g) >= 3 for g in groups.values):
                stat, p = kruskal(*groups.values)
                rows.append({"subset": label, "meta": meta, "lfp_feature": lfp,
                             "test": "Kruskal-Wallis", "stat": stat, "p": p})

    for lfp in LFP_NUMERICAL:
        d = df_sub[df_sub["lfp_feature"] == lfp].dropna(subset=["cort_depth","max_abs_cohens_d"])
        if len(d) >= 5:
            r, p = spearmanr(d["cort_depth"], d["max_abs_cohens_d"])
            rows.append({"subset": label, "meta": "cort_depth", "lfp_feature": lfp,
                        "test": "Spearman", "stat": r, "p": p})

df_meta_lfp = pd.DataFrame(rows)
_, df_meta_lfp["p_fdr"], _, _ = multipletests(
    df_meta_lfp["p"].fillna(1), method="fdr_bh")
df_meta_lfp["sig"] = df_meta_lfp["p_fdr"].apply(
    lambda p: "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns")

sig = df_meta_lfp[df_meta_lfp["sig"] != "ns"]
print(f"{len(sig)} significant metadata → LFP effect associations (BH-FDR < 0.05):")
display(sig.sort_values("p_fdr"))

# Visualise significant ones
for _, row in sig.iterrows():
    d = df_lfp_summary[df_lfp_summary["lfp_feature"] == row["lfp_feature"]].dropna(
        subset=[row["meta"], "max_abs_cohens_d"])
    fig, ax = plt.subplots(figsize=(6, 4))
    if row["test"] == "Kruskal-Wallis":
        order = sorted(d[row["meta"]].unique())
        sns.boxplot(data=d, x=row["meta"], y="max_abs_cohens_d", order=order,
                    palette=_CB_PALETTE[:len(order)], width=0.5, fliersize=0,
                    boxprops={"alpha": 0.5}, ax=ax)
        sns.stripplot(data=d, x=row["meta"], y="max_abs_cohens_d", order=order,
                      palette=_CB_PALETTE[:len(order)], alpha=0.7, jitter=0.15,
                      size=5, legend=False, ax=ax)
    else:
        ax.scatter(d["cort_depth"], d["max_abs_cohens_d"], alpha=0.4, s=20)
        ax.set_xlabel("Cortical Depth (µm)", fontsize=_FS_AX, fontweight="bold")
    ax.set_title(f"{row['meta']} → {row['lfp_feature']} [{row['subset']}]\n"
                 f"{row['test']}: stat={row['stat']:.2f}, p_fdr={row['p_fdr']:.3f} {row['sig']}",
                 fontsize=_FS_SUB)
    ax.set_ylabel("max |Cohen's d|", fontsize=_FS_AX, fontweight="bold")
    sns.despine(ax=ax)
    plt.tight_layout(); plt.show()

## Does waveform difference predict LFP effect strength?

**Numerical × Numerical**: Spearman ρ between nRMSE/cos_sim and max |Cohen's d|, separately for each LFP feature + BH-FDR across features.

Hypothesis: cells with bigger waveform differences between spike clusters should show stronger LFP modulation around those spikes.

In [ ]:
from scipy.stats import spearmanr

# Merge spike waveform metrics with LFP effect sizes
df_spk_base = (df_master_meta[["cell_id","spike_feature","nRMSE","cos_sim","temporal_rho"]]
               .drop_duplicates()
               .assign(spike_feature_base=lambda d: d["spike_feature"].str.replace("_cluster","",regex=False)))

df_merged = df_lfp_summary.merge(
    df_spk_base, on=["cell_id","spike_feature_base"], how="left")

for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label} (n={len(cell_ids)})\n{'='*60}")
    df_sub = df_merged[df_merged["cell_id"].isin(cell_ids)]

    rows = []
    for spk_m in ["nRMSE", "cos_sim"]:
        for lfp in LFP_NUMERICAL:
            d = df_sub[df_sub["lfp_feature"]==lfp].dropna(subset=[spk_m,"max_abs_cohens_d"])
            if len(d) < 5: continue
            r, p = spearmanr(d[spk_m], d["max_abs_cohens_d"])
            rows.append({"spike_metric": spk_m, "lfp_feature": lfp, "rho": r, "p": p, "n": len(d)})

    if not rows: continue
    df_r = pd.DataFrame(rows)
    _, df_r["p_fdr"], _, _ = multipletests(df_r["p"].fillna(1), method="fdr_bh")
    df_r["sig"] = df_r["p_fdr"].apply(
        lambda p: "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Waveform Difference → LFP Effect Strength — {label}",
                 fontsize=_FS_TTL, fontweight="bold")
    for ax, spk_m in zip(axes, ["nRMSE", "cos_sim"]):
        pivot = df_r[df_r["spike_metric"]==spk_m].set_index("lfp_feature")[["rho","sig"]]
        colors = [_SIG_COL if s != "ns" else _INSIG_COL for s in pivot["sig"]]
        bars = ax.barh(pivot.index, pivot["rho"], color=colors, alpha=0.8)
        ax.axvline(0, color="gray", lw=0.8)
        for bar, (_, row_r) in zip(bars, pivot.iterrows()):
            if row_r["sig"] != "ns":
                ax.text(row_r["rho"] + 0.01 * np.sign(row_r["rho"]),
                        bar.get_y() + bar.get_height()/2,
                        row_r["sig"], va="center", fontsize=10)
        ax.set_xlabel("Spearman ρ", fontsize=_FS_AX, fontweight="bold")
        ax.set_title(spk_m, fontsize=_FS_SUB, fontweight="bold")
        ax.set_yticklabels([f.replace("band_aucs.","") for f in pivot.index])
        sns.despine(ax=ax)
    plt.tight_layout(); plt.show()
    display(df_r.sort_values("p_fdr"))

## Pre/post LFP effects by metadata

**Categorical metadata × numerical pre/post effect** → Kruskal-Wallis + BH-FDR.

Do PCs vs INs show different pre-spike vs post-spike LFP modulation?

In [ ]:
rows = []
for label, cell_ids in SUBSET_LABELS:
    df_sub = df_prepost_meta[df_prepost_meta["cell_id"].isin(cell_ids)]
    for win, es_col in [("within","cohens_dz"), ("pre","hedges_g"), ("post","hedges_g")]:
        df_w = df_sub[df_sub["window"]==win].dropna(subset=[es_col])
        for meta in cat_meta:
            for lfp in LFP_NUMERICAL:
                d = df_w[df_w["lfp_feature"]==lfp].dropna(subset=[meta, es_col])
                groups = d.groupby(meta)[es_col].apply(list)
                if len(groups) >= 2 and all(len(g) >= 3 for g in groups.values):
                    stat, p = kruskal(*groups.values)
                    rows.append({"subset": label, "meta": meta, "lfp": lfp,
                                 "window": win, "stat": stat, "p": p})

df_pp_meta = pd.DataFrame(rows)
if not df_pp_meta.empty:
    _, df_pp_meta["p_fdr"], _, _ = multipletests(df_pp_meta["p"].fillna(1), method="fdr_bh")
    df_pp_meta["sig"] = df_pp_meta["p_fdr"].apply(
        lambda p: "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns")
    sig_pp = df_pp_meta[df_pp_meta["sig"] != "ns"]
    print(f"{len(sig_pp)} significant metadata → pre/post LFP associations:")
    display(sig_pp.sort_values("p_fdr"))
else:
    print("No data.")